Question 4: Statistical Analysis and Error Handling

Using scipy and the cleaned dataset from Question 1, perform the following tasks:

1. Implement error handling for the following analyses:
   - Perform a one-way ANOVA test to compare prices across different categories
   - Calculate and plot the confidence intervals for mean prices in each category
   - Identify potential outliers using z-scores

2. Debug and fix the following code snippet that attempts to perform a chi-square test:
   ```python
   def perform_chi_square(data):
       observed = data.groupby(['category', 'status']).size()
       chi2, p_value = stats.chi2_contingency(observed)
       return chi2, p_value
   ```

3. Implement proper logging to track:
   - Any statistical assumptions violations
   - Data type mismatches
   - Invalid calculations

Your solution should be robust against various edge cases and include appropriate error messages.

In [1]:
import pandas as pd
import numpy as np
from scipy import stats

import sys
import traceback

from sklearn.preprocessing import StandardScaler

In [2]:
sales_df = pd.read_csv('../data/curated/sales_data.csv')

In [3]:
# ANOVA test

# first, we collect prices for different categories
prices = {}
variances = []
epslon = 1
for category in sales_df['category'].unique():
    p = sales_df[sales_df['category'] == category]['price']
    prices[category] = p.values
    variances.append(np.var(p.values))

try:
    # then we need to check if data is normalized and variances are close to each other
    # we can run this check through a shapiro test
    for category in prices:
        statistic, pvalue = stats.shapiro(prices[category])
        assert pvalue > 0.05, f"Prices for category {category} are not normally distributed"
    # check if variances are close to each other
    for i in range(0,len(variances)):
        for j in range(i, len(variances)):
            assert variances[i] - variances[j] < epslon, f"The difference for two variances is higher than {epslon}" 
         
    anova = stats.f_oneway(*list(prices.values()))
except AssertionError:
    normalized_prices = {}
    _, _, tb = sys.exc_info()
    traceback.print_tb(tb) # Fixed format
    tb_info = traceback.extract_tb(tb)
    filename, line, func, text = tb_info[-1]
    print('An error occurred on line {} in statement {}'.format(line, text))
    print("Normalizing samples...")
    # Initialize the scaler for normalization
    for category in prices:
        normalized_prices[category] = (prices[category] - prices[category].mean())/prices[category].std()
    anova = stats.f_oneway(*list(normalized_prices.values()))

# honestly, I don't remember how to interpret the results of f_oneway

An error occurred on line 21 in statement assert variances[i] - variances[j] < epslon, f"The difference for two variances is higher than {epslon}"
Normalizing samples...


  File "/var/folders/j5/hg1r80jn575b35gd5nkvr_780000gn/T/ipykernel_25487/4269712937.py", line 21, in <module>
    assert variances[i] - variances[j] < epslon, f"The difference for two variances is higher than {epslon}"


In [4]:
# outlier detection
outliers = {}
for category in prices:
    outliers[category] = []
    mean = np.mean(prices[category])
    std = np.std(prices[category])
    for price in prices[category]:
        z_score = (price-mean)/std
        if z_score <= -3 or z_score >= 3:
            outliers[category].append(price)

outliers

{'Home & Garden': [],
 'Sports': [13.83213360155052],
 'Books': [],
 'Clothing': [],
 'Electronics': [195.79322703534584]}

In [5]:
# Chi-square test is used to test the dependency between two or more categorical variables.
# The example code mentions a column called "status", which can't be found in the data.
# I assumed here that 'status' meant  'quantity', which can be interpreted as a categorical value.

def perform_chi_square(data, alpha=0.05, print_conclusion=False):
    observed = data.groupby(['category', 'quantity'])[['sale_id']].count().to_numpy().transpose()
    total_values = len(observed[0])
    categories = len(sales_df['category'].unique())
    chi2, p_value, dof, expected = stats.chi2_contingency(observed.reshape(categories, total_values//categories))
    if print_conclusion:
        if p_value <= alpha:
            print("Dependent variables. Null hypothesis should be rejected.")
        else:
            print("Independent variables. Null hypothesis should be considered true.")
    return chi2, p_value, dof, observed

In [6]:
chi2, p_value, dof, observed = perform_chi_square(sales_df)